In [ ]:
# 1. Загрузите только basketball-cv.zip
import os, shutil, subprocess, sys, zipfile
from pathlib import Path
from google.colab import files

uploaded = files.upload()
archives = [name for name in uploaded if name.lower().endswith('.zip')]
if len(archives) != 1:
    raise ValueError('Загрузите ровно один файл: basketball-cv.zip')

archive_path = Path(archives[0])
extract_dir = Path('/content/_basketball_cv_extract')
PROJECT = Path('/content/basketball-cv')

# Чистая временная распаковка исключает случайное basketball-cv/basketball-cv
shutil.rmtree(extract_dir, ignore_errors=True)
extract_dir.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(archive_path) as archive:
    for member in archive.infolist():
        target = (extract_dir / member.filename).resolve()
        if extract_dir != target and extract_dir not in target.parents:
            raise ValueError('Некорректный путь внутри ZIP')
    archive.extractall(extract_dir)

# Находим корень проекта независимо от того, есть ли верхняя папка в ZIP.
markers = list(extract_dir.rglob('scripts/bootstrap_colab.py'))
if len(markers) != 1:
    raise AssertionError('Это не архив проекта Basketball CV: scripts/bootstrap_colab.py не найден')
source_root = markers[0].parent.parent

shutil.rmtree(PROJECT, ignore_errors=True)
if source_root == extract_dir:
    PROJECT.mkdir(parents=True, exist_ok=True)
    for item in list(extract_dir.iterdir()):
        shutil.move(str(item), str(PROJECT / item.name))
else:
    shutil.move(str(source_root), str(PROJECT))

shutil.rmtree(extract_dir, ignore_errors=True)
try:
    archive_path.unlink()
except OSError:
    pass

PYTHON = sys.executable
print('Проект загружен:', PROJECT)
print('Bootstrap найден:', PROJECT / 'scripts/bootstrap_colab.py')


In [ ]:
# 2. Быстрая установка. Основное приложение использует Python/PyTorch Colab.
# Если Colab уже на Python 3.13+, только Roboflow-worker автоматически получает Python 3.12.
# Пакеты Colab не откатываются до старых версий; ставится только недостающее.
# SAM2 и наши jersey_resnet.pt подключаются автоматически.
import time
from google.colab import userdata

def run_visible(command, **kwargs):
    environment = dict(os.environ, MPLBACKEND='Agg')
    environment.update(kwargs.pop('env', {}))
    with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, bufsize=1, env=environment, **kwargs) as process:
        for line in process.stdout:
            print(line, end='', flush=True)
        code = process.wait()
    if code:
        raise subprocess.CalledProcessError(code, command)

secrets_env = {}
for secret in ('ROBOFLOW_API_KEY', 'JERSEY_RESNET_URL'):
    try:
        value = userdata.get(secret)
        if value:
            secrets_env[secret] = value
            print(f'{secret}: найден')
    except Exception:
        pass

if 'ROBOFLOW_API_KEY' not in secrets_env:
    print('Внимание: добавьте ROBOFLOW_API_KEY в Colab Secrets перед анализом реального видео.')

print('Python Colab:', sys.version.split()[0])
started = time.time()
run_visible([
    PYTHON, '-u', str(PROJECT / 'scripts/bootstrap_colab.py'),
    '--device', 'cuda', '--roboflow', '--sam2'
], cwd=PROJECT, env=secrets_env)
print(f'Общая подготовка: {(time.time() - started) / 60:.1f} мин')


In [ ]:
# 3. Запуск интерфейса. Откройте ссылку *.gradio.live из вывода.
run_visible([
    PYTHON, '-u', '-m', 'basketball_cv.cli', 'ui', '--share'
], cwd=PROJECT, env=secrets_env)
